# Esercizio 4
### Analisi dati con Spark RDD
Il dataset `prodotti` contiene i dati dei prodotti in vendita:
- un identificativo univoco (`item_id`)
- la categoria del prodotto (`category`)
- il prezzo del singolo prodotto (`price`)

Il dataset `venditori` contiene i dati dei venditori:
- un identificativo univoco (`seller_id`)
- il tipo del venditore (`seller_type`), che può essere First-Party o Third-Party

Il dataset `transazioni` contiene i dati delle vendite effettuate:
- id univoco della transazione (`transaction_id`)
- id univoco del prodotto venduto (`item_id`)
- id univoco del venditore (`seller_id`)
- anno, mese e giorno della transazione (`year`, `month` e `day`)
- quantità di prodotti venduti (`num_items`)

In [2]:
import findspark
findspark.init()

#importing pyspark
import pyspark
#importing sparksession
from pyspark.sql import SparkSession

#creating a sparksession object and providing appName
spark=SparkSession.builder.appName("local").getOrCreate()
sc = spark.sparkContext

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/21 08:29:01 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
transazioni = spark.read.csv('../vendite/transazioni.csv', header=True, inferSchema=True).rdd.cache()
prodotti = spark.read.csv('../vendite/prodotti.csv', header=True, inferSchema=True).rdd.cache()
venditori = spark.read.csv('../vendite/venditori.csv', header=True, inferSchema=True).rdd.cache()

In [7]:
print( transazioni.take(3))
print( prodotti.take(3))
print( venditori.take(3))

[Row(transaction_id=1, item_id=1, seller_id=1, year=2009, month=5, day=29, num_items=1), Row(transaction_id=2, item_id=2, seller_id=8, year=2009, month=11, day=30, num_items=1), Row(transaction_id=3, item_id=3, seller_id=15, year=2009, month=1, day=27, num_items=1)]
[Row(item_id=1, category='Health & Personal Care', price=18.09), Row(item_id=2, category='Books', price=15.68), Row(item_id=3, category='Books', price=13.57)]
[Row(seller_id=1, seller_type='Third-Party'), Row(seller_id=8, seller_type='Third-Party'), Row(seller_id=15, seller_type='First-Party')]


In [15]:
#Per ogni mese, restituire la quantità venduta (`num_items`) massima in una singola transazione.

month_num_items = transazioni.map(lambda row: (row.month, row.num_items))
month_num_items.reduceByKey(max).sortByKey().collect() 



[(1, 3),
 (2, 3),
 (3, 15),
 (4, 2),
 (5, 2),
 (6, 3),
 (7, 2),
 (8, 3),
 (9, 2),
 (10, 16),
 (11, 3),
 (12, 5)]

In [16]:
#Calcolare il ricavo ottenuto da ogni transazione, moltiplicando il prezzo del prodotto e la quantità venduta.

tran = transazioni.map(lambda row: (row.item_id, row ))
prod = prodotti.map(lambda row: (row.item_id, row ))

result = prod.join(tran).values().cache()
print(result.collect())

final_result = result.map(
    lambda row: (row[1].transaction_id , row[0].price * row[1].num_items )
)

final_result.take(3)

[(Row(item_id=2, category='Books', price=15.68), Row(transaction_id=2, item_id=2, seller_id=8, year=2009, month=11, day=30, num_items=1)), (Row(item_id=4, category='Books', price=23.1), Row(transaction_id=4, item_id=4, seller_id=13, year=2009, month=12, day=21, num_items=1)), (Row(item_id=6, category='Electronics', price=59.99), Row(transaction_id=6, item_id=6, seller_id=7, year=2009, month=3, day=17, num_items=1)), (Row(item_id=8, category='Electronics', price=3.33), Row(transaction_id=8, item_id=8, seller_id=5, year=2009, month=7, day=3, num_items=2)), (Row(item_id=10, category='Books', price=11.97), Row(transaction_id=10, item_id=10, seller_id=5, year=2009, month=11, day=30, num_items=1)), (Row(item_id=12, category='Kitchen & Housewares', price=2.29), Row(transaction_id=12, item_id=12, seller_id=7, year=2009, month=10, day=21, num_items=1)), (Row(item_id=14, category='MP3 Downloads', price=1.99), Row(transaction_id=14, item_id=14, seller_id=11, year=2009, month=10, day=31, num_items

[(2, 15.68), (4, 23.1), (6, 59.99)]

In [18]:
#Calcolare il ricavo massimo ottenuto da ciascun venditore.

ricavo_veditore = result.map(
    lambda row: ( row[1].seller_id, row[0].price * row[1].num_items )
)

ricavo_veditore.reduceByKey(max).sortByKey().collect()

[(1, 119.99),
 (2, 180.0),
 (3, 214.18),
 (4, 130.0),
 (5, 107.09),
 (6, 107.09),
 (7, 107.09),
 (8, 107.09),
 (9, 829.0),
 (10, 829.0),
 (11, 179.99),
 (12, 147.38),
 (13, 359.0),
 (14, 107.09),
 (15, 359.0),
 (16, 109.99),
 (17, 214.18),
 (18, 242.99),
 (19, 106.31),
 (20, 299.99)]

In [22]:
# Calcoliamo il ricavo totale per ciascun venditore

ricavo_totale = ricavo_veditore.reduceByKey( lambda x,y : x+y)
#ricavo_totale.sortByKey().collect()
# Ordiniamo le chiavi e arrotondiamo i valori
ricavo_totale.mapValues(lambda x: round(x, 2)).sortByKey().collect()

[(1, 1206.58),
 (2, 1289.63),
 (3, 1312.22),
 (4, 1213.1),
 (5, 812.48),
 (6, 1041.86),
 (7, 854.68),
 (8, 906.3),
 (9, 2263.64),
 (10, 1695.16),
 (11, 2702.78),
 (12, 2805.15),
 (13, 3598.16),
 (14, 2238.86),
 (15, 3477.58),
 (16, 2654.43),
 (17, 2816.42),
 (18, 2285.16),
 (19, 2640.56),
 (20, 2369.13)]

In [8]:
 #Calcolare il ricavo medio ottenuto da ciascun venditore.



# Trasformazioni per ordinare un generico Pair RDD: mapValues + reduceByKey + mapValues
def avg_by_key(pair_rdd):
    return pair_rdd.mapValues(lambda x: (x, 1)).reduceByKey(sum_values_counters).mapValues(calc_avg)

def sum_values_counters(x, y):
    value1, count1 = x
    value2, count2 = y
    return value1 + value2, count1 + count2

def calc_avg(sum_values_counter):
    sum_values, counter = sum_values_counter
    return sum_values / counter

ricavi_medi = avg_by_key(ricavo_veditore).mapValues(lambda x: round(x, 2))
ricavi_medi.sortByKey().collect()

NameError: name 'ricavo_veditore' is not defined

In [24]:
#Ordinare i risultati ottenuti nell'esercizio precedente in base al ricavo medio, in modo decrescente.
ricavi_medi.sortBy( lambda x: x[1], ascending=False).collect()

[(9, 51.45),
 (10, 42.38),
 (15, 35.13),
 (13, 34.93),
 (12, 31.52),
 (18, 30.47),
 (2, 29.99),
 (17, 29.96),
 (16, 27.94),
 (3, 26.78),
 (20, 26.62),
 (4, 26.37),
 (14, 25.73),
 (11, 25.26),
 (1, 24.62),
 (19, 24.23),
 (6, 23.68),
 (8, 22.66),
 (5, 21.96),
 (7, 18.58)]

In [29]:
#Calcolare i 10 venditori che hanno ottenuto la somma di ricavi maggiore nel mese di Maggio del 2009.

ricavo_veditore2 = result.map(
    lambda row: ( row[1].seller_id,row[1].month, row[1].year, row[0].price * row[1].num_items )
)

ricavo_veditore2 = ricavo_veditore2.filter( lambda x: x[1] == 5 and x[2] == 2009 ).map( lambda x: (x[0],x[3])).reduceByKey( lambda x,y: x+y)

ricavo_veditore2.top(10, lambda x: x[1])





[(15, 632.76),
 (12, 425.61),
 (17, 382.45),
 (18, 315.5),
 (20, 303.53),
 (11, 300.33),
 (4, 280.78),
 (8, 186.44),
 (14, 166.25),
 (19, 157.83999999999997)]

In [4]:
#Calcolare il ricavo medio ottenuto dai venditori "First-Party" per ogni categoria di prodotto.


# Le informazioni che ci servono sono le seguenti:
# `seller_type`, che si trova nell'RDD `venditori`
# `ricavo`, che si trova nell'RDD `prod_tr_ricavi`
# `category`, che si trova nell'RDD `prod_tr_ricavi`

tran = transazioni.map(lambda row: (row.item_id, row ))
prod = prodotti.map(lambda row: (row.item_id, row ))

result = prod.join(tran).values().cache()

# Filtriamo i venditori mantenendo solo quelli First-Party
venditori_first = venditori.filter(lambda row: row.seller_type == 'First-Party')
ricavo_veditore = result.map(
    lambda row: ( row[1].seller_id, row[0].price * row[1].num_items )
)

# Facciamo il join con `prod_tr_ricavi` in base al `seller_id`
seller_id_venditori_first = venditori_first.map(lambda row: (row.seller_id, row))
seller_id_prod_tr_ricavi = result.map(lambda x: (x['transazione'].seller_id, x))
seller_prod_tr_ricavi = seller_id_venditori_first.join(seller_id_prod_tr_ricavi).values().cache()

# Creiamo il pair rdd `(categoria, ricavo)`
categoria_ricavo = seller_prod_tr_ricavi.values().map(lambda x: (x['prodotto'].category, x['ricavo']))

# Calcoliamo la media per ogni categoria
(avg_by_key(categoria_ricavo)               # calcoliamo la media
    .mapValues(lambda x: round(x, 2))      # arrotondiamo i valori
    .sortByKey()                           # ordiniamo in base alla chiave
    .collectAsMap())     

NameError: name 'avg_by_key' is not defined

In [11]:
spark.read.csv('../vendite/transazioni.csv', header=True, inferSchema=True).createOrReplaceTempView('transazioni')
spark.read.csv('../vendite/prodotti.csv', header=True, inferSchema=True).createOrReplaceTempView('prodotti')
spark.read.csv('../vendite/venditori.csv', header=True, inferSchema=True).createOrReplaceTempView('venditori')

In [ ]:
#Per ogni mese, restituire la quantità venduta (`num_items`) massima in una singola transazione.

spark.sql('''
    SELECT month, max(num_items) AS max_num_items
    FROM transazioni
    GROUP BY month
    ORDER BY month
''').show()

+-----+-------------+
|month|max_num_items|
+-----+-------------+
|    1|            3|
|    2|            3|
|    3|           15|
|    4|            2|
|    5|            2|
|    6|            3|
|    7|            2|
|    8|            3|
|    9|            2|
|   10|           16|
|   11|            3|
|   12|            5|
+-----+-------------+



In [9]:
#Calcolare il ricavo ottenuto da ogni transazione, moltiplicando il prezzo del prodotto e la quantità venduta.

spark.sql(''' 
    SELECT tr.seller_id, pr.price * tr.num_items AS ricavo_totale
    FROM transazioni tr, prodotti pr
    WHERE tr.item_id = pr.item_id

''').show()

{"ts": "2026-01-21 08:33:04.554", "level": "ERROR", "logger": "SQLQueryContextLogger", "msg": "[TABLE_OR_VIEW_NOT_FOUND] The table or view `transazioni` cannot be found. Verify the spelling and correctness of the schema and catalog.\nIf you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.\nTo tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS. SQLSTATE: 42P01", "context": {"errorClass": "TABLE_OR_VIEW_NOT_FOUND"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o27.sql.\n: org.apache.spark.sql.AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The table or view `transazioni` cannot be found. Verify the spelling and correctness of the schema and catalog.\nIf you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.\nTo tolerate the error on drop use DROP VIEW IF EXISTS or DROP 

AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The table or view `transazioni` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS. SQLSTATE: 42P01; line 3 pos 9;
'Project ['tr.seller_id, ('pr.price * 'tr.num_items) AS ricavo_totale#78]
+- 'Filter ('tr.item_id = 'pr.item_id)
   +- 'Join Inner
      :- 'SubqueryAlias tr
      :  +- 'UnresolvedRelation [transazioni], [], false
      +- 'SubqueryAlias pr
         +- 'UnresolvedRelation [prodotti], [], false


In [10]:
#Calcolare il ricavo massimo ottenuto da ciascun venditore.

spark.sql(''' 
    SELECT tr.seller_id, max(pr.price * tr.num_items) AS ricavo_max
    FROM transazioni tr, prodotti pr
    WHERE tr.item_id = pr.item_id
    GROUP BY tr.seller_id

''').show()


{"ts": "2026-01-21 08:33:08.157", "level": "ERROR", "logger": "SQLQueryContextLogger", "msg": "[TABLE_OR_VIEW_NOT_FOUND] The table or view `transazioni` cannot be found. Verify the spelling and correctness of the schema and catalog.\nIf you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.\nTo tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS. SQLSTATE: 42P01", "context": {"errorClass": "TABLE_OR_VIEW_NOT_FOUND"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o27.sql.\n: org.apache.spark.sql.AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The table or view `transazioni` cannot be found. Verify the spelling and correctness of the schema and catalog.\nIf you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.\nTo tolerate the error on drop use DROP VIEW IF EXISTS or DROP 

AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The table or view `transazioni` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS. SQLSTATE: 42P01; line 3 pos 9;
'Aggregate ['tr.seller_id], ['tr.seller_id, 'max(('pr.price * 'tr.num_items)) AS ricavo_max#79]
+- 'Filter ('tr.item_id = 'pr.item_id)
   +- 'Join Inner
      :- 'SubqueryAlias tr
      :  +- 'UnresolvedRelation [transazioni], [], false
      +- 'SubqueryAlias pr
         +- 'UnresolvedRelation [prodotti], [], false


In [13]:
#Calcoliamo il ricavo totale per ciascun venditore

spark.sql(''' 
    SELECT tr.seller_id, sum(pr.price * tr.num_items) AS ricavo_tot
    FROM transazioni tr, prodotti pr
    WHERE tr.item_id = pr.item_id
    GROUP BY tr.seller_id

''').show()


+---------+------------------+
|seller_id|        ricavo_tot|
+---------+------------------+
|       12|           2805.15|
|        1|1206.5800000000004|
|       13|3598.1599999999994|
|        6|1041.8600000000001|
|       16|2654.4299999999994|
|        3|           1312.22|
|       20|2369.1299999999987|
|        5| 812.4800000000001|
|       19|2640.5599999999977|
|       15| 3477.579999999999|
|       17|           2816.42|
|        9|2263.6399999999994|
|        4|1213.1000000000001|
|        8| 906.3000000000001|
|        7|            854.68|
|       10|1695.1600000000003|
|       11| 2702.779999999998|
|       14|2238.8599999999997|
|        2|1289.6300000000003|
|       18|2285.1599999999985|
+---------+------------------+



In [15]:
#Calcolare il ricavo medio ottenuto da ciascun venditore.
 
spark.sql(''' 
SELECT tr.seller_id, avg(pr.price * tr.num_items) AS ricavo_avg
FROM transazioni tr, prodotti pr
WHERE tr.item_id = pr.item_id
GROUP BY tr.seller_id

''').show()

+---------+------------------+
|seller_id|        ricavo_avg|
+---------+------------------+
|       12|  31.5185393258427|
|        1| 24.62408163265307|
|       13|34.933592233009705|
|        6|23.678636363636368|
|       16|27.941368421052626|
|        3|             26.78|
|       20|26.619438202247178|
|        5|21.958918918918922|
|       19| 24.22532110091741|
|       15|  35.1270707070707|
|       17| 29.96191489361702|
|        9| 51.44636363636362|
|        4|26.371739130434786|
|        8|22.657500000000002|
|        7|             18.58|
|       10|42.379000000000005|
|       11| 25.25962616822428|
|       14|25.734022988505743|
|        2|29.991395348837216|
|       18| 30.46879999999998|
+---------+------------------+



In [17]:
#Ordinare i risultati ottenuti nell'esercizio precedente in base al ricavo medio, in modo decrescente.

spark.sql(''' 
SELECT tr.seller_id, avg(pr.price * tr.num_items) AS ricavo_avg
FROM transazioni tr, prodotti pr
WHERE tr.item_id = pr.item_id
GROUP BY tr.seller_id
ORDER BY ricavo_avg DESC

''').show()

+---------+------------------+
|seller_id|        ricavo_avg|
+---------+------------------+
|        9| 51.44636363636362|
|       10|42.379000000000005|
|       15|  35.1270707070707|
|       13|34.933592233009705|
|       12|  31.5185393258427|
|       18| 30.46879999999998|
|        2|29.991395348837216|
|       17| 29.96191489361702|
|       16|27.941368421052626|
|        3|             26.78|
|       20|26.619438202247178|
|        4|26.371739130434786|
|       14|25.734022988505743|
|       11| 25.25962616822428|
|        1| 24.62408163265307|
|       19| 24.22532110091741|
|        6|23.678636363636368|
|        8|22.657500000000002|
|        5|21.958918918918922|
|        7|             18.58|
+---------+------------------+



In [12]:
#Calcolare il ricavo medio ottenuto dai venditori "First-Party" per ogni categoria di prodotto.


spark.sql("""
    SELECT
        pr.category,
        AVG(pr.price * tr.num_items) AS ricavo_medio
    FROM transazioni tr
    JOIN prodotti pr
        ON tr.item_id = pr.item_id
    JOIN venditori v
        ON tr.seller_id = v.seller_id
    WHERE v.seller_type = 'First-Party'
    GROUP BY pr.category
""").show()


+--------------------+------------------+
|            category|      ricavo_medio|
+--------------------+------------------+
|    Tools & Hardware|28.747999999999998|
|Apparel & Accesso...|             31.99|
|       MP3 Downloads|3.8983333333333334|
|Health & Personal...|             22.49|
|                 DVD|19.727600000000006|
|        Pet Supplies|20.740000000000002|
|        Toys & Games|             14.87|
|Kitchen & Housewares| 41.63600000000001|
|                Baby|            29.985|
|   Sports & Outdoors|13.203999999999999|
|         Electronics|37.173947368421054|
|         Video Games| 34.48833333333334|
|       Kindle eBooks| 6.680384615384619|
|               Other|1.9900000000000002|
|          Automotive|              9.37|
|               Books|30.138349928876163|
|Health & Personal...|31.650000000000002|
|     Kindle Hardware|217.99571428571429|
|Video On Demand V...|1.9900000000000002|
|Magazine Subscrip...|             19.95|
+--------------------+------------